# 06 - Binary NLI Fine-Tuning (ContraDoc)

Fine-tune `cross-encoder/nli-deberta-v3-base` for binary contradiction detection on
the leakage-controlled splits prepared in `nli_finetune_data.ipynb`.

The pretrained checkpoint has 3 NLI classes (contradiction / entailment / neutral). We
replace the classification head with a 2-class head (`0 = not_contradiction`, `1 = contradiction`)
and let `ignore_mismatched_sizes=True` reinitialize it. Encoder weights transfer; the head re-trains.

**Inputs (held-out, base_doc_id-disjoint):**
- `data/processed/ContraDoc/nli/nli_train.csv`
- `data/processed/ContraDoc/nli/nli_val.csv`
- `data/processed/ContraDoc/nli/nli_test.csv`

**Output:** `../fine-tuning/models/nli_binary/` (gitignored)


In [ ]:
import csv
import random
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import classification_report
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

NLI_DIR = Path("data/processed/ContraDoc/nli")
TRAIN_CSV = NLI_DIR / "nli_train.csv"
VAL_CSV   = NLI_DIR / "nli_val.csv"
TEST_CSV  = NLI_DIR / "nli_test.csv"

MODEL_NAME = "cross-encoder/nli-deberta-v3-base"
OUTPUT_DIR = Path("../fine-tuning/models/nli_binary")
SEED = 42
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if device.type == "cuda":
    print(f"gpu:    {torch.cuda.get_device_name(0)}")

## Load splits

Splits were built in `nli_finetune_data.ipynb` with strict `base_doc_id` disjointness
between train / val / test. Re-assert that here so any future regeneration is caught.

In [ ]:
def load_pairs(csv_path):
    rows = []
    with csv_path.open(encoding="utf-8") as f:
        for r in csv.DictReader(f):
            rows.append({
                "premise": r["premise"],
                "hypothesis": r["hypothesis"],
                "label": int(r["label"]),
                "base_doc_id": r["base_doc_id"],
                "contra_type": r["contra_type"],
                "kind": r["kind"],
            })
    return rows

train_pairs = load_pairs(TRAIN_CSV)
val_pairs   = load_pairs(VAL_CSV)
test_pairs  = load_pairs(TEST_CSV)

def stats(name, pairs):
    pos = sum(p["label"] for p in pairs)
    print(f"{name:6s}: {len(pairs):4d} pairs  (pos={pos}, neg={len(pairs)-pos})")

stats("train", train_pairs)
stats("val",   val_pairs)
stats("test",  test_pairs)

# leakage check
train_bases = {p["base_doc_id"] for p in train_pairs}
val_bases   = {p["base_doc_id"] for p in val_pairs}
test_bases  = {p["base_doc_id"] for p in test_pairs}
assert not (train_bases & val_bases),  f"train/val overlap: {train_bases & val_bases}"
assert not (train_bases & test_bases), f"train/test overlap: {train_bases & test_bases}"
assert not (val_bases & test_bases),   f"val/test overlap: {val_bases & test_bases}"
print(f"
base_doc_id disjoint: train={len(train_bases)}, val={len(val_bases)}, test={len(test_bases)}")

## Tokenizer + model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)
model.config.id2label = {0: "not_contradiction", 1: "contradiction"}
model.config.label2id = {"not_contradiction": 0, "contradiction": 1}

assert model.classifier.weight.shape[0] == 2, "Head not binary"
print(f"params: {sum(p.numel() for p in model.parameters()):,}")
print(f"head:   {tuple(model.classifier.weight.shape)}")

## Datasets

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len):
        enc = tokenizer(
            [p["premise"]    for p in pairs],
            [p["hypothesis"] for p in pairs],
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt",
        )
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.token_type_ids = enc.get("token_type_ids")
        self.labels         = torch.tensor([p["label"] for p in pairs], dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {
            "input_ids":      self.input_ids[i],
            "attention_mask": self.attention_mask[i],
            "labels":         self.labels[i],
        }
        if self.token_type_ids is not None:
            item["token_type_ids"] = self.token_type_ids[i]
        return item

train_ds = PairDataset(train_pairs, tokenizer, MAX_LEN)
val_ds   = PairDataset(val_pairs,   tokenizer, MAX_LEN)
test_ds  = PairDataset(test_pairs,  tokenizer, MAX_LEN)
print(f"train_ds={len(train_ds)}  val_ds={len(val_ds)}  test_ds={len(test_ds)}")

## Train

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

trainer.train()

## Validation report

In [ ]:
val_out = trainer.predict(val_ds)
y_pred = np.argmax(val_out.predictions, axis=1)
y_true = [p["label"] for p in val_pairs]
print(classification_report(
    y_true, y_pred,
    target_names=["not_contradiction", "contradiction"],
    digits=3,
))

## Test report (held-out)

This is the only place we touch the test split. The 150 balanced docs that produced
these test pairs have base_doc_ids that never appear in train or val.

In [ ]:
test_out = trainer.predict(test_ds)
y_pred = np.argmax(test_out.predictions, axis=1)
y_true = [p["label"] for p in test_pairs]
print(classification_report(
    y_true, y_pred,
    target_names=["not_contradiction", "contradiction"],
    digits=3,
))

## Save model

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"saved to {OUTPUT_DIR.resolve()}")
print("files:", sorted(f.name for f in OUTPUT_DIR.iterdir()))